# TODO

- Finalize reward function — current `self_efficacy - stress` is a placeholder, may
      optimize for rest/leisure over productive behavior
- Run PPO training and evaluate learned policy
- Decide whether traits re-randomize on every `reset()` or stay fixed per episode
      (affects training diversity vs. controlled experiments)
- Document openness — currently only touches leisure slightly, needs either a stronger
      mapping or an honest note that it operates at policy level not environment level


# Synthetic student journal RL model
This notebook trains a model to simulate a Fontys ICT student's psychological state using reinforcent learning techiniques.

### Observations / environment (the student's internal state)
These are the variables the environment tracks at each timestep (one simulated day). They represent the student's psychological state.

Each variable is a value between 0 and 1.

- `energy` — Lowers through cognitively demanding actions (studying, attending lectures)
  and restores through rest or leisure.
  
  Models ego depletion: a student who attends lectures all day has less capacity to study in the evening. Bottoming out on energy
  amplifies the negative effects of high-stress actions.

- `stress` — Rises passively as deadlines approach and spikes when a deadline is missed.
  Falls through rest, leisure, or completing study actions while not overwhelmed. Maps to the
  Yerkes-Dodson law: moderate stress can improve performance (studying while mildly stressed
  is productive), but high stress (>0.8) reverses the effect of studying.

- `self_efficacy` — Shifts slowly based on recent behavior. Rises through productive
  actions (studying, attending lectures) and falls through avoidance (skipping) or studying
  while overwhelmed.

- `social_need` — Rises passively each day regardless of actions. Only meaningfully
  reduced by socializing or leisure.

- `days_until_deadline (normalized)` — External pressure signal, stored internally as an
  integer and normalized for the observation space. Resets on a 10-day cycle right now. Drives the passive
  stress tick and deadline-miss penalty.

---

### Actions (what the agent can do each day)
One action is taken per timestep. Effects are applied after passive state ticks, meaning
deadline pressure and social drift accumulate before the action resolves.

- `study` — Costs energy. If stress < 0.8: reduces stress, increases self_efficacy.
  If stress > 0.8: energy cost increases, stress rises further, and self_efficacy drops.
  Models the Yerkes-Dodson breakdown under high load.

- `attend lecture` — Moderate energy cost. Small stress reduction and self_efficacy gain.
  Lower impact than studying but without the high-stress penalty, a reliable low-risk action.

- `leisure` — Restores energy, reduces stress, slightly reduces social_need. Minor self_efficacy
  penalty, reflecting time not spent on productive tasks. The balanced recovery option.

- `socialize` — Small energy cost, significant social_need reduction, minor self_efficacy gain.
  No direct stress effect. High-extraversion personas benefit more from this action (see Persona).

- `rest` — Strong energy restoration, moderate stress reduction. No effect on self_efficacy or
  social_need. The pure recovery action, high value when energy is critically low.

- `skip / do nothing` — Small energy gain (no cognitive load). Stress increases slightly,
  self_efficacy drops. Short-term relief at the cost of longer-term confidence and deadline pressure.
---

### Traits (Static layer of persona)
These are fixed per-student parameters ranging from 0–1, based on the Big Five personality model.
Unlike the observation variables, traits do not change during an episode. These traits shape how different student personas respond to the same situation differently.

- `Openness` — Low: prefers routine, sticks to familiar study habits, resistant to trying
  new approaches. High: experiments with different methods, more likely to attend optional events
  or explore unfamiliar topics.

- `Conscientiousness` — Low: impulsive, easily distracted, avoids tasks under pressure.
  High: reliable and persistent, more likely to study even when stressed, stronger deadline
  response. Directly modulates how stress affects the decision to study vs. skip.

- `Extraversion` — Low: socializing is draining rather than restorative, strong preference
  for solo rest. High: energy partially restored by social activity, social_need depletes faster
  when isolated. Shifts the energy cost/benefit of the socialize action.
  
- `Agreeableness` — Low: socially selective, less motivated by group work or peer pressure.
  High: values harmony, more likely to attend lectures and group sessions, social interactions
  restore more social_need. Influences how relatedness (SDT) is weighted.

- `Neuroticism` — Low: stress stays stable under moderate pressure, recovers quickly.
  High: stress spikes harder near deadlines, self_efficacy drops faster after failures, and the
  passive stress tick is amplified. Most directly tied to the Yerkes-Dodson dynamics in the env.

#### How traits interact with the environment
Traits don't change what actions are available, they change what actions 'cost and return'.
A high-conscientiousness student who studies while stressed takes less of an energy hit than a
low-conscientiousness student doing the same. A high-extraversion student who socializes recovers
more energy. This means the same reward-maximizing policy looks behaviorally different across
personas, which is what makes the synthetic journal entries feel distinct for different traits in personas.

#### Locus of Control (supplementary trait)
Outside the Big Five, locus of control (internal vs. external, 0–1) is worth including as it
shapes journal language rather than the states. An internal student writes "I decided to
skip today", an external student writes "the lecture was pointless so I didn't go." This is a useful
signal for the LLM generation step even if it doesn't affect the RL state directly.

### Reward (What should the agent work towards)
The reward function is not yet finalized — this is a placeholder to get the environment running. For now we use reward = self_efficacy - stress, which loosely optimizes for a student who feels capable and not overwhelmed. This may be revised once we have a clearer picture of what the model should actually be learning to do.

---

## Changes to the environment

### General rules that run every step regardless of action:
- days_until_deadline ticks down by 1
- stress increases passively if deadline is close: stress += 0.1 * (1 - days_until_deadline/max_days)
- social_need drifts up passively each day: social_need = min(1, social_need + 0.05)

### Per action:
**study**
- energy -= 0.2 (cognitive load)
- stress -= 0.15 (making progress)
- self_efficacy += 0.05 (competence building)

**attend lecture**
- energy -= 0.15 (less than studying, more passive)
- stress -= 0.05 (slight relief, staying on track)
- self_efficacy += 0.03 (competence, but slower than active studying)

**leisure**
- energy += 0.15 (restores somewhat, less than full rest)
- stress -= 0.15 
- social_need -= 0.1
- self_efficacy -= 0.01 (very slight guilt/opportunity cost, barely noticeable)

**socialize**
- energy -= 0.1
- social_need -= 0.3 
- self_efficacy += 0.02 (relatedness boost, small)

**rest**
- energy += 0.3
- stress -= 0.1

**skip / do nothing**
- energy += 0.1 (short term relief)
- stress += 0.05 (guilt/falling behind)
- self_efficacy -= 0.05

---

## Dependencies

In [15]:
import gymnasium as gym
import numpy as np

print("Gymnasium version:", gym.__version__)
print("NumPy version:", np.__version__)

Gymnasium version: 1.3.0
NumPy version: 1.26.4


## Intialize the environment

In [16]:
class StudentEnv(gym.Env):
    def __init__(self, traits=None):
        super(StudentEnv, self).__init__()
        self.action_space = gym.spaces.Discrete(6)  # Study, Attend lecture, Leisure, Socialize, Rest, Skip

        # Accept explicit traits or randomize for diverse training data
        if traits is not None:
            self.traits = traits
        else:
            self.traits = {
                'openness':          np.random.uniform(0, 1),
                'conscientiousness': np.random.uniform(0, 1),
                'extraversion':      np.random.uniform(0, 1),
                'agreeableness':     np.random.uniform(0, 1),
                'neuroticism':       np.random.uniform(0, 1)
            }
        
        self.timestep = 0
        self.max_days = 30
        self.days_until_deadline = 10
        self.days_deadline = 10

        # Observation: [energy, stress, self_efficacy, social_need, days_until_deadline]
        self.observation_space = gym.spaces.Box(
            low=0, 
            high=1, 
            shape=(5,), 
            dtype=np.float32
        )
        
        self.state = np.array([
            0.5,
            0.5,
            0.5,
            0.5,
            self.days_until_deadline / self.max_days
        ], dtype=np.float32)      

    def step(self, action):
        energy, stress, self_efficacy, social_need, _ = self.state

        o = self.traits['openness']
        c = self.traits['conscientiousness']
        e = self.traits['extraversion']
        a = self.traits['agreeableness']
        n = self.traits['neuroticism']

        # Passive ticks
        # Neuroticism amplifies deadline-driven stress
        stress += 0.1 * (1 - self.days_until_deadline / self.days_deadline) * (1 + 0.5 * n)
        social_need = min(1, social_need + 0.05)
        self.days_until_deadline = max(0, self.days_until_deadline - 1)
        
        if self.days_until_deadline == 0:
            self.days_until_deadline = 10
            stress = min(1, stress + 0.2 * (1 + 0.5 * n))  # neurotic students spiral harder on missed deadlines

        if action == 0:  # Study
            # Conscientiousness raises the breakdown threshold, neuroticism lowers it
            stress_threshold = 0.8 + 0.1 * c - 0.1 * n
            if stress > stress_threshold:
                energy -= 0.3 * (1 + 0.2 * n)        # neurotic: higher energy cost when overwhelmed
                stress += 0.05
                self_efficacy -= 0.05 * (1 + 0.3 * n) # neurotic: confidence takes a harder hit
            else:
                energy -= 0.2 * (1 - 0.2 * c)         # conscientious: studies more efficiently
                stress -= 0.15
                self_efficacy += 0.05

        elif action == 1:  # Attend lecture
            # Agreeableness: more engaged in structured group settings
            energy -= 0.15
            stress -= 0.05 * (1 + 0.2 * a)            # agreeable: finds lectures more relieving
            self_efficacy += 0.03 * (1 + 0.2 * a)     # agreeable: gets more out of the social learning context

        elif action == 2:  # Leisure
            # Openness: more variety in leisure, slightly more restorative
            energy += 0.15 * (1 + 0.1 * o)
            stress -= 0.15
            social_need -= 0.1
            self_efficacy -= 0.01

        elif action == 3:  # Socialize
            # Extraversion: socializing costs less and restores more social_need
            energy -= 0.1 * (1 - 0.4 * e)             # introverts: socializing is more draining
            social_need -= 0.3 * (1 + 0.3 * e)        # extraverts: get more out of social contact
            self_efficacy += 0.02

        elif action == 4:  # Rest
            # Conscientiousness: slight guilt during rest, marginally less restorative
            energy += 0.3
            stress -= 0.1 * (1 - 0.1 * (1 - c))       # low conscientiousness: rest relieves stress more easily

        elif action == 5:  # Skip
            # Conscientiousness: skipping hurts self-efficacy more for conscientious students
            energy += 0.1
            stress += 0.05 * (1 + 0.2 * n)            # neurotic: more guilt from avoidance
            self_efficacy -= 0.05 * (1 + 0.3 * c)     # conscientious: skipping conflicts with self-image

        energy = np.clip(energy, 0, 1)
        stress = np.clip(stress, 0, 1)
        self_efficacy = np.clip(self_efficacy, 0, 1)
        social_need = np.clip(social_need, 0, 1)

        self.state = np.array([
            energy,
            stress,
            self_efficacy,
            social_need,
            self.days_until_deadline / self.max_days
        ], dtype=np.float32)

        reward = self_efficacy - stress
        self.timestep += 1
        terminated = False
        truncated = self.timestep >= self.max_days

        return self.state, reward, terminated, truncated, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Re-randomize traits each episode for diverse training data
        self.traits = {
            'openness':          np.random.uniform(0, 1),
            'conscientiousness': np.random.uniform(0, 1),
            'extraversion':      np.random.uniform(0, 1),
            'agreeableness':     np.random.uniform(0, 1),
            'neuroticism':       np.random.uniform(0, 1)
        }
        self.days_until_deadline = 10
        self.state = np.array([
            0.5,
            0.5,
            0.5,
            0.5,
            self.days_until_deadline / self.max_days
        ], dtype=np.float32)
        self.timestep = 0
        return self.state, {}

## Running the model for 30 days

In [17]:
import requests
import os

os.makedirs("synthetic_journals", exist_ok=True)

ACTION_NAMES = ["study", "attend lecture", "leisure", "socialize", "rest", "skip"]
max_days = 30

env = StudentEnv()
obs, _ = env.reset()

prev_obs = None

for day in range(max_days):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)

    energy, stress, self_efficacy, social_need, days_norm = obs
    days_until_deadline = int(days_norm * max_days)
    action_name = ACTION_NAMES[action]

    prev_section = ""
    if prev_obs is not None:
        pe, ps, pse, psn, pd = prev_obs
        prev_section = f"""Yesterday:
  energy: {pe:.2f}, stress: {ps:.2f}, self_efficacy: {pse:.2f}, social_need: {psn:.2f}
"""

    prompt = f"""You are a student writing in your personal diary. Write a short, honest diary entry for today. Stay grounded — only describe things that match the event and state below. No invented events.

{prev_section}Today (day {env.timestep}):
  Action taken: {action_name}
  energy: {energy:.2f}
  stress: {stress:.2f}
  self_efficacy: {self_efficacy:.2f}
  social_need: {social_need:.2f}
  days until deadline: {days_until_deadline}
  reward signal: {reward:+.2f}

Write the diary entry in first person. 3-5 sentences. Reflect the emotional tone the state implies."""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma4:e4b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.0
            }
        }
    )

    diary_text = response.json()["response"].strip()

    filename = f"synthetic_journals/day_{env.timestep:02d}.txt"
    with open(filename, "w") as f:
        f.write(f"Day {env.timestep} | Action: {action_name}\n")
        f.write(f"State: energy={energy:.2f}, stress={stress:.2f}, self_efficacy={self_efficacy:.2f}, social_need={social_need:.2f}, deadline in {days_until_deadline}d\n")
        f.write("-" * 60 + "\n")
        f.write(diary_text + "\n")

    print(f"Day {env.timestep:02d} written — {action_name}")
    prev_obs = obs

    if done:
        break

print("Done — journals saved to synthetic_journals/")

Day 01 written — attend lecture
Day 02 written — skip
Day 03 written — attend lecture
Day 04 written — leisure
Day 05 written — skip
Day 06 written — attend lecture
Day 07 written — attend lecture
Day 08 written — study
Day 09 written — socialize
Day 10 written — socialize
Day 11 written — study
Day 12 written — socialize
Day 13 written — skip
Day 14 written — skip
Day 15 written — attend lecture
Day 16 written — attend lecture
Day 17 written — leisure
Day 18 written — attend lecture
Day 19 written — attend lecture
Day 20 written — leisure
Day 21 written — attend lecture
Day 22 written — rest
Day 23 written — study
Day 24 written — leisure
Day 25 written — study
Day 26 written — socialize
Day 27 written — leisure
Day 28 written — study
Day 29 written — skip
Day 30 written — attend lecture
Done — journals saved to synthetic_journals/
